# EU Cost-of-Living Analysis: Price Levels vs. Net Earnings

This notebook compares the **Price Level Index (PLI)** for household consumption with **net earnings**
across EU27 countries, to see whether higher-earning countries are also higher-priced ones, and to build
an **affordability index** (earnings adjusted for local price levels).

**Data sources (Eurostat):**
- `eurostat_prc_ppp_ind_1_2025_pli_e011.csv` — Price Level Index, household final consumption (EU27 = 100)
- `earn_nt_net_earnings_euro.csv` — Annual net earnings at 100% of national average wage
- `earn_nt_net__earnings_50.csv` — Annual net earnings at 50% of national average wage

**Sections:**
1. Setup & reusable helper functions
2. Price Level Index (PLI) — load, validate, explore
3. Net earnings (100% of average) — load, validate, explore, merge with PLI
4. Net earnings (50% of average) — same pipeline, for a lower-income comparison
5. Comparing affordability at 100% vs. 50% of average earnings
6. Export processed data


In [ ]:
import sys
print(sys.executable)
print(sys.version)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Environment ready!")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## 1. Reusable helper functions

All the repeated logic (loading, validating, merging, computing the affordability index, plotting) is
factored into functions here. They are written generically so they can be copy-pasted into other
projects with minimal changes.

In [ ]:
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

# Eurostat aggregate/non-EU codes to drop when we only want the 27 member states
NON_EU27_GEO_CODES = ["CH", "IS", "NO", "EA21"]  # Switzerland, Iceland, Norway, Euro area aggregate
EU27_CODE = "EU27_2020"  # Eurostat code for the EU27 aggregate row

COUNTRY_NAME_COL = "Geopolitical entity (reporting)"
GEO_COL = "geo"
VALUE_COL = "OBS_VALUE"

In [ ]:
def list_raw_files(raw_dir: Path) -> None:
    """Print each file in raw_dir with its size, so you can sanity-check what's available."""
    for file in raw_dir.iterdir():
        print(file.name, "-", file.stat().st_size, "bytes")


def load_csv_with_preview(path: Path, n_preview_lines: int = 10, encoding: str = "utf-8") -> pd.DataFrame:
    """
    Print the first `n_preview_lines` of a CSV (useful for eyeballing structure/encoding issues
    before pandas parses it), then load and return the full file as a DataFrame.
    """
    with open(path, "r", encoding=encoding) as file:
        for _ in range(n_preview_lines):
            line = file.readline()
            if not line:
                break
            print(line.rstrip())
    return pd.read_csv(path)


def check_data_quality(df: pd.DataFrame, id_col: str, value_col: str) -> None:
    """Print duplicate-id and missing-value counts, the two most basic sanity checks for this dataset."""
    n_duplicates = df[id_col].duplicated().sum()
    n_missing = df[value_col].isna().sum()
    print(f"Duplicate '{id_col}' values: {n_duplicates}")
    print(f"Missing '{value_col}' values: {n_missing}")


def exclude_geo_codes(df: pd.DataFrame, codes_to_exclude: list, geo_col: str = GEO_COL) -> pd.DataFrame:
    """Return a copy of df with the given geo codes (e.g. non-EU countries or an aggregate row) removed."""
    return df[~df[geo_col].isin(codes_to_exclude)].copy()


def get_reference_value(df: pd.DataFrame, geo_col: str, geo_value: str, value_col: str) -> float:
    """Look up a single scalar value for one geo code, e.g. the EU27 aggregate's OBS_VALUE."""
    return df.loc[df[geo_col] == geo_value, value_col].iloc[0]

In [ ]:
def fit_linear_regression(x: pd.Series, y: pd.Series) -> dict:
    """
    Fit y = slope * x + intercept by least squares (equivalent to a single-variable OLS regression).
    Returns slope, intercept, fitted values, residuals, Pearson correlation and R-squared in one call,
    so the fit only has to be computed once and reused everywhere it's needed.
    """
    slope, intercept = np.polyfit(x, y, 1)
    predicted = slope * x + intercept
    residuals = y - predicted
    correlation = np.corrcoef(x, y)[0, 1]
    r_squared = correlation ** 2
    return {
        "slope": slope,
        "intercept": intercept,
        "predicted": predicted,
        "residuals": residuals,
        "correlation": correlation,
        "r_squared": r_squared,
    }


def compute_affordability_index(
    df: pd.DataFrame,
    pli_col: str,
    earnings_col: str,
    eu27_earnings: float,
    suffix: str = "",
) -> pd.DataFrame:
    """
    Add three derived columns to a copy of df:
      - earnings_index{suffix}:      earnings as a % of the EU27 reference (EU27 = 100)
      - earnings_to_pli{suffix}:     earnings divided by the price level index (purchasing-power proxy)
      - affordability_index{suffix}: earnings_to_pli rescaled so the EU27 average = 100,
                                      i.e. how far a country's "real" earning power is from the EU27 norm
    `suffix` lets you compute this twice in the same notebook (e.g. "_50" for the 50%-earner scenario)
    without column name clashes.
    """
    df = df.copy()
    df[f"earnings_index{suffix}"] = (df[earnings_col] / eu27_earnings) * 100
    df[f"earnings_to_pli{suffix}"] = df[earnings_col] / df[pli_col]
    df[f"affordability_index{suffix}"] = (df[f"earnings_to_pli{suffix}"] / (eu27_earnings / 100)) * 100
    return df

In [ ]:
def plot_ranked_bar(
    categories,
    values,
    xlabel: str,
    ylabel: str,
    title: str,
    reference_line: float = None,
    reference_label: str = None,
    color=None,
    figsize=(15, 8),
):
    """Horizontal bar chart, optionally with a vertical reference line (e.g. an index baseline of 100)."""
    fig, ax = plt.subplots(figsize=figsize)
    ax.barh(categories, values, color=color)

    if reference_line is not None:
        ax.axvline(reference_line, color="red", linestyle="--", linewidth=2, label=reference_label)
        ax.legend()

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def plot_scatter_with_trend(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    label_col: str,
    xlabel: str,
    ylabel: str,
    title: str,
    subtitle: str = None,
    eu27_x: float = 100,
    eu27_y: float = None,
    eu27_y_label: str = None,
    color: str = "tab:blue",
    figsize=(12, 8),
) -> dict:
    """
    Scatter plot of x vs y with each point labelled, a fitted linear-trend line, and optional EU27
    reference lines on both axes. Returns the regression dict from `fit_linear_regression` so the
    caller can reuse the fit (e.g. for a residual analysis) instead of refitting it.
    """
    fig, ax = plt.subplots(figsize=figsize)
    fig.suptitle(title, fontsize=18, fontweight="bold")
    if subtitle:
        ax.set_title(subtitle, fontsize=12, pad=10)

    x, y = df[x_col], df[y_col]
    ax.scatter(x, y, color=color)

    for _, row in df.iterrows():
        ax.annotate(row[label_col], (row[x_col], row[y_col]), xytext=(5, 5), textcoords="offset points")

    reg = fit_linear_regression(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = reg["slope"] * x_line + reg["intercept"]
    ax.plot(x_line, y_line, color="black", linewidth=2, label="Linear trend")

    if eu27_x is not None:
        ax.axvline(eu27_x, color="red", linestyle="--", linewidth=1.5, label=f"EU27 PLI = {eu27_x}")
    if eu27_y is not None:
        label = eu27_y_label or f"EU27 reference = {eu27_y:,.2f}"
        ax.axhline(eu27_y, color="green", linestyle="--", linewidth=1.5, label=label)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend()
    plt.show()

    return reg

## 2. Price Level Index (PLI)

### 2.1 Load raw data

In [ ]:
list_raw_files(RAW_DATA_DIR)

In [ ]:
pli_file_path = RAW_DATA_DIR / "eurostat_prc_ppp_ind_1_2025_pli_e011.csv"

# df_pli_raw: unfiltered PLI data straight from Eurostat, one row per geo (country/aggregate)
df_pli_raw = load_csv_with_preview(pli_file_path)

In [ ]:
df_pli_raw.shape

In [ ]:
df_pli_raw.head()

In [ ]:
df_pli_raw.dtypes

In [ ]:
df_pli_raw.info()

In [ ]:
df_pli_raw[[GEO_COL, COUNTRY_NAME_COL, VALUE_COL]]

In [ ]:
# df_pli: PLI data restricted to the EU27 member states + the EU27 aggregate row
# (drops Switzerland, Iceland, Norway and the Euro-area-21 aggregate, which aren't EU27 members)
df_pli = exclude_geo_codes(df_pli_raw, NON_EU27_GEO_CODES)
df_pli.shape

In [ ]:
df_pli[GEO_COL].tolist()

### 2.2 Data validation / quality check

In [ ]:
check_data_quality(df_pli, id_col=GEO_COL, value_col=VALUE_COL)

Check the year / indicator / expenditure category are all consistent (i.e. we're not accidentally mixing years or categories):

In [ ]:
df_pli["TIME_PERIOD"].unique()

In [ ]:
df_pli["indic_ppp"].unique()

In [ ]:
df_pli["ppp_cat18"].unique()

### 2.3 Descriptive statistics

In [ ]:
# df_pli_countries: the 27 member states only, excluding the EU27 aggregate row itself
df_pli_countries = exclude_geo_codes(df_pli, [EU27_CODE])
df_pli_countries.shape

In [ ]:
df_pli_countries[VALUE_COL].describe()

In [ ]:
df_pli[VALUE_COL].describe()  # includes the EU27 aggregate, so the stats are skewed relative to the country-only view above

### 2.4 Bar chart

In [ ]:
df_pli_countries_sorted = df_pli_countries.sort_values(VALUE_COL, ascending=True)

plot_ranked_bar(
    categories=df_pli_countries_sorted[COUNTRY_NAME_COL],
    values=df_pli_countries_sorted[VALUE_COL],
    xlabel="Price Level Index (EU27 = 100)",
    ylabel="Country",
    title="Household Consumption Price Levels in EU Countries, 2025",
    reference_line=100,
    reference_label="EU27 reference = 100",
)

### 2.5 Analytics: spread between the cheapest and most expensive country

In [ ]:
min_pli = df_pli_countries[VALUE_COL].min()
max_pli = df_pli_countries[VALUE_COL].max()

lowest_pli_country = df_pli_countries.loc[df_pli_countries[VALUE_COL].idxmin(), COUNTRY_NAME_COL]
highest_pli_country = df_pli_countries.loc[df_pli_countries[VALUE_COL].idxmax(), COUNTRY_NAME_COL]

# How much pricier the most expensive country is, relative to the cheapest (cheapest as the base)
spread_from_min = (max_pli - min_pli) / min_pli * 100
# How much cheaper the cheapest country is, relative to the most expensive (most expensive as the base)
spread_from_max = (max_pli - min_pli) / max_pli * 100

print(f"Lowest PLI:  {lowest_pli_country} ({min_pli:.1f})")
print(f"Highest PLI: {highest_pli_country} ({max_pli:.1f})")
print(f"The most expensive country is {spread_from_min:.1f}% pricier than the cheapest.")
print(f"The cheapest country is {spread_from_max:.1f}% cheaper than the most expensive.")

### 2.6 Conclusions — PLI dataset

- 27 EU countries, EU27 reference retained separately
- No duplicate country observations, no missing PLI values
- Consistent year, indicator and expenditure category throughout
- See the printed summary above for the cheapest/most expensive country and the spread between them


## 3. Net earnings — 100% of national average

### 3.1 Load raw data

In [ ]:
list_raw_files(RAW_DATA_DIR)

In [ ]:
earnings_100_file_path = RAW_DATA_DIR / "earn_nt_net_earnings_euro.csv"

# df_earnings_100_raw: annual net earnings (EUR) at 100% of the national average wage, one row per geo
df_earnings_100_raw = load_csv_with_preview(earnings_100_file_path)

In [ ]:
df_earnings_100_raw.shape

In [ ]:
df_earnings_100_raw.info()

In [ ]:
df_earnings_100_raw[[GEO_COL, COUNTRY_NAME_COL, VALUE_COL]]

### 3.2 Data validation / quality check

In [ ]:
df_earnings_100_raw.columns.tolist()

In [ ]:
check_data_quality(df_earnings_100_raw, id_col=GEO_COL, value_col=VALUE_COL)

In [ ]:
df_earnings_100_raw["TIME_PERIOD"].unique()

In [ ]:
df_earnings_100_raw["Earnings structure"].unique()

In [ ]:
df_earnings_100_raw["Earnings case"].unique()

In [ ]:
df_earnings_100_raw["currency"].unique()

### 3.3 Descriptive statistics

In [ ]:
df_earnings_100_raw[VALUE_COL].describe()

In [ ]:
# eu27_earnings_100: EU27 average net earnings (EUR) at 100% of national average wage — used
# throughout as the reference point for indices below
eu27_earnings_100 = get_reference_value(df_earnings_100_raw, GEO_COL, EU27_CODE, VALUE_COL)

# df_earnings_100_countries: the 27 member states only, excluding the EU27 aggregate row
df_earnings_100_countries = exclude_geo_codes(df_earnings_100_raw, [EU27_CODE])

print(f"EU27 average net earnings (100%): €{eu27_earnings_100:,.2f}")
df_earnings_100_countries[VALUE_COL].describe()

### 3.4 Merge PLI and earnings

In [ ]:
set(df_pli_countries[GEO_COL]) == set(df_earnings_100_countries[GEO_COL])  # confirm both datasets cover the same 27 countries

In [ ]:
pli_for_merge = df_pli_countries[[GEO_COL, COUNTRY_NAME_COL, VALUE_COL]].rename(columns={VALUE_COL: "pli"})
earnings_100_for_merge = df_earnings_100_countries[[GEO_COL, VALUE_COL]].rename(columns={VALUE_COL: "net_earnings_eur"})

# df_merged_100: one row per country with PLI and 100%-of-average net earnings side by side
df_merged_100 = pli_for_merge.merge(earnings_100_for_merge, on=GEO_COL, how="inner")
# how="inner" keeps only geo codes present in both datasets;
# use "left"/"right" to keep everything from one side, or "outer" to keep everything from both
df_merged_100.shape

In [ ]:
df_merged_100.head()

### 3.5 Plots and stats — are higher-earning countries also higher-priced?

In [ ]:
regression_100 = plot_scatter_with_trend(
    df_merged_100,
    x_col="pli",
    y_col="net_earnings_eur",
    label_col=GEO_COL,
    xlabel="Price Level Index (EU27 = 100)",
    ylabel="Annual net earnings (€)",
    title="Price Levels and Net Earnings in EU Countries, 2025",
    subtitle="Net earnings at 100% of national average",
    eu27_y=eu27_earnings_100,
    eu27_y_label=f"EU27 net earnings = €{eu27_earnings_100:,.2f}",
)

correlation_100 = regression_100["correlation"]
r_squared_100 = regression_100["r_squared"]
print(f"Pearson correlation: {correlation_100:.3f}")
print(f"R-squared: {r_squared_100:.3f}")

### 3.6 Residual analysis

Which countries earn more (or less) than their price level alone would predict?

In [ ]:
df_merged_100["predicted_earnings"] = regression_100["predicted"]
df_merged_100["earnings_residual"] = regression_100["residuals"]
print("earnings_residual(i) = net_earnings_eur(i) - predicted_earnings(i)")

df_merged_100.sort_values("earnings_residual", ascending=False)[
    [GEO_COL, "pli", "net_earnings_eur", "predicted_earnings", "earnings_residual"]
]

### 3.7 Affordability index

In [ ]:
df_merged_100 = compute_affordability_index(
    df_merged_100, pli_col="pli", earnings_col="net_earnings_eur", eu27_earnings=eu27_earnings_100
)

df_merged_100.sort_values("affordability_index", ascending=True)[
    [GEO_COL, "pli", "net_earnings_eur", "earnings_to_pli", "affordability_index"]
]

In [ ]:
df_affordability_100_sorted = df_merged_100.sort_values("affordability_index", ascending=True)

plot_ranked_bar(
    categories=df_affordability_100_sorted[GEO_COL],
    values=df_affordability_100_sorted["affordability_index"],
    xlabel="Earnings-to-Price-Level Affordability Index",
    ylabel="Country",
    title="Net earnings at 100% of national average",
    reference_line=100,
    reference_label="EU27 reference = 100",
)

## 4. Net earnings — 50% of national average

Same pipeline as Section 3, applied to earners at 50% of the national average wage, so we can see
whether affordability patterns hold up for lower incomes too.

### 4.1 Load raw data

In [ ]:
earnings_50_file_path = RAW_DATA_DIR / "earn_nt_net__earnings_50.csv"

# df_earnings_50_raw: annual net earnings (EUR) at 50% of the national average wage, one row per geo
df_earnings_50_raw = pd.read_csv(earnings_50_file_path)
df_earnings_50_raw.shape

In [ ]:
df_earnings_50_raw.info()

### 4.2 Descriptive statistics

In [ ]:
df_earnings_50_raw[VALUE_COL].describe()

In [ ]:
# eu27_earnings_50: EU27 average net earnings (EUR) at 50% of national average wage
eu27_earnings_50 = get_reference_value(df_earnings_50_raw, GEO_COL, EU27_CODE, VALUE_COL)

# df_earnings_50_countries: the 27 member states only, excluding the EU27 aggregate row
df_earnings_50_countries = exclude_geo_codes(df_earnings_50_raw, [EU27_CODE])

print(f"EU27 average net earnings (50%): €{eu27_earnings_50:,.2f}")
df_earnings_50_countries.shape

In [ ]:
df_earnings_50_countries[[GEO_COL, COUNTRY_NAME_COL, VALUE_COL]].sort_values(VALUE_COL)

### 4.3 Merge with PLI

In [ ]:
earnings_50_for_merge = df_earnings_50_countries[[GEO_COL, VALUE_COL]].rename(columns={VALUE_COL: "net_earnings_50_eur"})

# df_merged_50: one row per country with PLI and 50%-of-average net earnings side by side
df_merged_50 = df_merged_100[[GEO_COL, "pli"]].merge(earnings_50_for_merge, on=GEO_COL, how="inner")
df_merged_50.shape

### 4.4 Plot: earnings vs. PLI at 50% of average earnings

In [ ]:
regression_50 = plot_scatter_with_trend(
    df_merged_50,
    x_col="pli",
    y_col="net_earnings_50_eur",
    label_col=GEO_COL,
    xlabel="Price Level Index (EU27 = 100)",
    ylabel="Annual net earnings (€) at 50%",
    title="Price Levels and Net Earnings in EU Countries, 2025",
    subtitle="Net earnings at 50% of national average",
    eu27_y=eu27_earnings_50,
    eu27_y_label=f"EU27 net earnings = €{eu27_earnings_50:,.2f}",
    color="darkorange",
)

correlation_50 = regression_50["correlation"]
r_squared_50 = regression_50["r_squared"]
print(f"Pearson correlation: {correlation_50:.3f}")
print(f"R-squared: {r_squared_50:.3f}")

### 4.5 Residual analysis

In [ ]:
df_merged_50["predicted_earnings_50"] = regression_50["predicted"]
df_merged_50["earnings_residual_50"] = regression_50["residuals"]

### 4.6 Affordability index (50%)

In [ ]:
df_merged_50 = compute_affordability_index(
    df_merged_50, pli_col="pli", earnings_col="net_earnings_50_eur", eu27_earnings=eu27_earnings_50, suffix="_50"
)

df_merged_50.sort_values("affordability_index_50", ascending=True)[
    [GEO_COL, "pli", "net_earnings_50_eur", "earnings_to_pli_50", "affordability_index_50"]
]

In [ ]:
df_affordability_50_sorted = df_merged_50.sort_values("affordability_index_50", ascending=True)

plot_ranked_bar(
    categories=df_affordability_50_sorted[GEO_COL],
    values=df_affordability_50_sorted["affordability_index_50"],
    xlabel="Earnings-to-Price-Level Affordability Index",
    ylabel="Country",
    title="Net earnings at 50% of national average",
    reference_line=100,
    reference_label="EU27 reference = 100",
)

## 5. Comparing affordability at 100% vs. 50% of average earnings

In [ ]:
# df_comparison: one row per country with both the 100%-earner and 50%-earner metrics side by side
df_comparison = df_merged_100.merge(
    df_merged_50.drop(columns=["pli"]),  # "pli" is identical in both frames, so drop the duplicate before merging
    on=GEO_COL,
    how="inner",
)

df_comparison["affordability_change"] = (
    df_comparison["affordability_index_50"] - df_comparison["affordability_index"]
)

df_comparison.info()

In [ ]:
df_comparison[
    [GEO_COL, "affordability_index", "affordability_index_50", "affordability_change"]
].sort_values("affordability_change", ascending=False)

### Plot: dumbbell chart of affordability at 100% vs 50%

In [ ]:
df_comparison_sorted = df_comparison.sort_values("affordability_change").copy()

fig, ax = plt.subplots(figsize=(10, 12))
fig.suptitle("Affordability: 100% vs 50% of Average Earnings", fontsize=18, fontweight="bold")

y_positions = range(len(df_comparison_sorted))

for i, (_, row) in enumerate(df_comparison_sorted.iterrows()):
    # Connecting line between the two scenarios for this country
    ax.plot(
        [row["affordability_index"], row["affordability_index_50"]],
        [i, i],
        color="gray", linewidth=2, alpha=0.6,
    )
    ax.scatter(
        row["affordability_index"], i, color="steelblue", s=70,
        label="100% average earnings" if i == 0 else "",
    )
    ax.scatter(
        row["affordability_index_50"], i, color="darkorange", s=70,
        label="50% average earnings" if i == 0 else "",
    )

ax.set_yticks(list(y_positions))
ax.set_yticklabels(df_comparison_sorted[GEO_COL])
ax.axvline(100, color="black", linestyle="--", linewidth=1.5, alpha=0.7)  # EU27 benchmark

ax.set_xlabel("Affordability Index (EU27 = 100)")
ax.set_ylabel("Country")
ax.legend()

plt.tight_layout()
plt.show()

## 6. Export processed data

In [ ]:
PROCESSED_DATA_DIR.mkdir(exist_ok=True)
output_path = PROCESSED_DATA_DIR / "eu27_earnings_affordability_final.csv"

df_comparison.to_csv(output_path, index=False)
print(f"Saved: {output_path}")